# GPU 程序计时与测量：方法总结
> 适用：测任何 GPU 代码的耗时与显存（算子、层、训练步、推理）。
> 一句话：**GPU 是异步的，所以必须同步、预热、重复、控制变量，并且先有预测再去测。**
> 另一句：**要测训练步的各阶段，就在同一步里用 event 切段，不要用不同运行的结果相减。**

---

## 1. 为什么不能直接 `time.time()`

| 问题 | 机理 | 后果 |
|---|---|---|
| 异步发射 | Python 调用只是把 kernel 放进队列就返回 | 不同步量到的是发射时间，偏小几十倍 |
| 首次迭代非稳态 | cuBLAS 初始化、kernel 懒加载、allocator 首次分配、`torch.compile` 编译、autotune | 首次比稳态慢几倍到几百倍 |
| 频率与温度 | GPU 随负载、温度调频 | 前后几次不一致 |
| 其他进程 | 卡不空闲 | 抖动大，数据不可信 |
| 缓存状态 | 微基准反复用同一份输入，L2 常驻 | 带宽测得偏乐观 |

---

## 2. 通用六步流程
1. **定义被测单元**：无参函数 `fn()`，只含要测的内容。数据生成、H2D、`.item()`、打印都放在外面。
   **先想清楚它带不带计算图**：`no_grad` 下的 forward、带图的 forward、backward、optimizer.step 是四个不同的单元，成本和显存都不一样（见 4A）。
2. **固定条件**：设备独占、dtype、形状、种子、TF32 开关、`train/eval`、`no_grad` 与否。
3. **预热**：不计时地跑若干次，让初始化、编译、allocator、频率进入稳态。
4. **同步后计时**：CUDA event，或计时前后各 `synchronize`。
5. **重复并统计**：中位数为主，同时记最小、最大、抖动。作业要求“均值±std”时同时报均值、样本 std 和中位数。
6. **算派生指标，对照预测**：TFLOP/s、GB/s、tokens/s，与理论上限或预测比。

---

## 3. 关键 API

| API | 用途 | 注意 |
|---|---|---|
| `torch.cuda.synchronize()` | 等 GPU 上已发射的工作全部完成 | 预热结束、计时边界必须调用 |
| `torch.cuda.Event(enable_timing=True)` | GPU 侧时间戳 | `e.record()` 入当前 stream；读 `elapsed_time` 前保证 `e` 已完成。同一步里可以 record 多个 event，把一步切成几段 |
| `time.perf_counter()` | CPU 墙钟 | 仅在前后同步时有意义，用于连续吞吐 |
| `torch.no_grad()` / `inference_mode()` | 关 autograd | 测纯 forward 必须用 |
| `opt.zero_grad(set_to_none=True)` | 清梯度 | 与真实训练一致 |
| `torch.cuda.reset_peak_memory_stats()` | 重置峰值计数 | 每个阶段开始前调用；放在预热之后 |
| `torch.cuda.max_memory_allocated()` | 张量实际占用峰值 | 对照理论估算 |
| `torch.cuda.max_memory_reserved()` | allocator 持有峰值 | 与上一项之差反映碎片 |
| `torch.utils.benchmark.Timer` | 官方计时器，自动预热与同步 | 快速测小函数 |
| `triton.testing.do_bench` | 微基准，默认清 L2 | 单 kernel 对比 |
| `torch.cuda.CUDAGraph` | 消除发射开销 | 区分“kernel 慢”与“发射慢” |
| `torch.autocast("cuda", dtype=torch.bfloat16)` | 混合精度上下文 | 只包 forward（见 4B） |
| `torch.set_float32_matmul_precision("highest"/"high")` 与 `backends.cuda.matmul.allow_tf32` | 开关 fp32 GEMM 是否走 TF32 | 每个配置开始前显式设置，并把读回的值记进结果 |
| `module.register_forward_hook` | 实测各模块输出 dtype | 按类名挂，不依赖属性名 |
| `statistics.stdev` | 样本标准差（n−1） | 样本只有十来个时，别用总体标准差 |

---

## 4. 两种计时方式，回答不同问题

| 方式 | 做法 | 回答 | 特点 |
|---|---|---|---|
| 逐次 event | 每次迭代前后各一个 event，每次之间同步 | 一步本身的 GPU 耗时（延迟） | 不含步间 CPU 空隙，稳定，适合与预测比 |
| 连续墙钟 | 连跑 N 步，仅头尾同步，总时间除以 N | 真实吞吐 | 含 CPU 发射与流水重叠，tokens/s 用它 |
两个数都记。`wall` 明显大于 `median` → 有 CPU 空隙（发射受限）；明显小于 → 步间有重叠。
要测训练步的 fwd / bwd / opt 三阶段，不要跑三档再相减，见 4A。

---

## 4A. 训练步的阶段计时：fwd / bwd / opt
### 4A.1 计算图决定了怎么测

| 事实 | 对测量的含义 |
|---|---|
| **forward 在 grad 模式下会建图**：每个算子记录一个节点，并保存反向需要的张量 | 带图的 forward 比 `no_grad` 的 forward 慢（CPU 要建节点），并且多占激活显存 |
| **`backward()` 沿图反向走，用完就释放保存的张量**（默认 `retain_graph=False`） | 同一个 loss 不能 backward 两次（会报错）。**每次计时迭代都必须重新 forward 一遍**，不能“建一次图反复测 backward” |
| `retain_graph=True` 可绕过 | 保存的张量一直占着显存，backward 的行为与真实训练不同，**不要用来测速** |
| **`no_grad` 下不建图**，中间张量用完即释放 | 测纯 forward（推理）用它；显存低得多。例：本项目 fwd 峰值 0.71 GiB，fwd+bwd 峰值 4.06 GiB |
| **`optimizer.step()` 不建图**，读 `p.grad` 更新参数与状态 | 标准 AdamW 的状态在**第一次 step 才分配**（2 份 fp32，约 8 B/参数），所以预热必须包含至少一次 step |
| **`zero_grad(set_to_none=True)`** 释放梯度，backward 时直接新建 | 少一次逐参数累加 kernel；benchmark 与 profile 脚本必须用同一种写法，否则 kernel 数对不上 |

由此得到各阶段的**正确定义**：

| 阶段 | 内容 | 图 |
|---|---|---|
| forward(+loss) | 带图的 forward，加 loss | 建图，保存激活 |
| backward | `loss.backward()` | 消耗并释放图 |
| optimizer | `opt.step()` | 无图 |
| （对照）no_grad forward | 纯前向，不含 loss | 无图 |

### 4A.2 “相减法”是什么，有什么问题
做法：分别跑三档，`backward ≈ fwdbwd − fwd`，`optimizer ≈ full − fwdbwd`。

| # | 问题 | 机理 |
|---|---|---|
| 1 | **减数和被减数不是同一种 forward** | `fwd` 是 `no_grad`（无图、无 loss），`fwdbwd` 里的 forward 带图、带 loss（还可能有 `logits.float()` 转换）。差值里混进了“建图加 loss 的成本”，**backward 被高估** |
| 2 | **用大数相减得小数，误差被放大** | 例：full 127.85，fwdbwd 114.90，optimizer = 12.95。若两个中位数各有约 ±1 ms 的波动，optimizer 的相对误差就是约 ±8%，而单看 full 只有不到 1% |
| 3 | **得不到各阶段自己的 std** | 三档是三次独立运行，没有“同一步的三段”可以逐步相减。只能近似 √(s₁²+s₂²)，这个近似还假设两次独立，而实际上它们共享同一个频率与温度状态 |
| 4 | **各档的运行状态不同** | `fwdbwd` 档从不调 `opt.step()`，所以没有 Adam 状态，显存基线和 allocator 状态与 `full` 不同；档与档之间 GPU 频率、温度也不同 |
| 5 | **作业要求的均值±std 无法直接给出** | 3.2 表每个阶段都要 mean±std |

相减法仍然有用的地方：**交叉核对**。三档的结果相减，应该与下面的阶段计时结果大体一致，差得多说明某一边有问题。

### 4A.3 解决办法（理论）：在同一步里用 event 切段
在一个训练步内放 4 个 event，**整步只在最后同步一次**：
```
zero_grad                        （只置 None，无 GPU 工作，放在 event 之外）
ev0.record()
  logits = model(x)  ;  loss = CE(logits.float(), y)      ← forward(+loss)，带图
ev1.record()
  loss.backward()                                          ← backward
ev2.record()
  opt.step()                                               ← optimizer
ev3.record()
synchronize()                                              ← 每步只同步一次
forward = ev0→ev1,  backward = ev1→ev2,  optimizer = ev2→ev3
```
为什么这样对：
- event 记录在 stream 里，间隔是 **GPU 时间线上的间隔**，包含 kernel 之间的空隙（发射受限时就是真实成本），但不包含步与步之间的 CPU 空隙。
- 阶段之间**没有额外同步**，不会破坏 CPU 与 GPU 的真实重叠（这是和 nsys 里每阶段加 `synchronize` 的区别，那个只用来做构成分析）。
- 三段加起来就是整步（`step = f + b + o`），不需要相减，每个阶段各有 n 个样本，可以直接算均值和 std。
- 每一步结束同步一次，所以步与步之间不重叠，这一步测的是“延迟”。想要“吞吐”，另外连续跑 N 步只同步一次（见第 4 节）。

### 4A.4 本方法的边界与对照

| 事项 | 说明 |
|---|---|
| forward 含 loss | 表头写清楚。要和 11.5 的 forward 预测对比，另外用 `no_grad` 的纯前向 |
| 峰值显存 | 在预热之后用单独的探针步测：反向结束时、完整步结束时。此时 Adam 状态已存在，所以**两个数都含 8 B/参数**；要测不含优化器状态的 fwd+bwd 峰值，用 `--no-opt` |
| 预算先行 | 完整训练步至少需要约 16 B/参数（权重 4、梯度 4、Adam 8）。large（本配置约 9.7 亿参数）光静态部分就约 15.5 GB（约 14.5 GiB），再加激活，在 16 GiB 的卡上 batch 4 **几乎肯定 OOM**（batch 很小时有可能放下，要实测）。OOM 不是 bug，应该在状态列写 OOM，再用 `--no-opt` 补 fwd+bwd。注意 GB 与 GiB 相差约 7%，估算时单位要统一 |
| 和 nsys 的关系 | nsys 里阶段末尾 `synchronize` 破坏重叠，只看构成；这里的阶段时间才是真实耗时 |

---

## 4B. 混合精度（bf16 autocast）测量
### 4B.1 autocast 设置指南

| 设置 | 做法 | 为什么 |
|---|---|---|
| **参数保持 fp32** | 模型本身不转 bf16 | fp32 主权重，避免 `lr × grad` 这类小更新被舍入掉（bf16 只有约 8 位有效数字） |
| **autocast 只包 forward** | `with torch.autocast("cuda", dtype=torch.bfloat16): logits = model(x)` | autocast 是对“算子调用”生效的，只需要包住前向 |
| **loss 在 autocast 之外、用 fp32** | `loss = CE(logits.float(), y)` | 对数、指数、求和在 bf16 里精度不够。`logits.float()` 是一次额外的类型转换，计入 forward |
| **backward 在 autocast 之外调用** | `loss.backward()` | 官方推荐。反向算子自动沿用对应前向算子的 dtype，不需要再包 |
| **bf16 不需要 GradScaler** | 不加 | bf16 的指数范围与 fp32 相同，不会下溢；GradScaler 是 fp16 才需要 |
| **TF32 显式关闭（fp32 对照组）** | `set_float32_matmul_precision("highest")` 与 `allow_tf32=False` | 否则 fp32 对照组可能悄悄走 TF32，加速比被低估 |
| **权重类型转换缓存** | 默认开启：同一个 autocast 区域内，权重只转换一次 | 每个训练步重新进入区域，所以每步都会重新转一遍权重，这是额外开销，计入 forward |
| **只测推理的超大模型** | `bf16_weights`：权重直接转 bf16，不用 autocast | 例：2.7B 在 fp32 + autocast 下要同时占 fp32 权重和 bf16 缓存，16 GB 卡放不下 |

### 4B.2 autocast 下各算子的 dtype 规则
按 PyTorch 官方文档的分类（具体清单以你的版本文档为准）：

| 类别 | 代表算子 | 行为 |
|---|---|---|
| 降到 bf16 | `linear`、`matmul`、`mm`、`bmm`、`addmm`、卷积 | 输入被转成 bf16，输出是 bf16。`einsum` 内部分解为 `bmm`，所以也走 bf16 |
| 升到 fp32 | `exp`、`log`、`softmax`、`log_softmax`、`sum`、`pow`、`layer_norm`、`cross_entropy` 等 | 输入被转成 fp32，输出 fp32 |
| 取输入中最宽的类型 | `addcmul`、`dot`、`index_put` 等 | |
| 其余（`add`、`mul`、`cat`、`where`、`relu` 等） | | **按输入的 dtype 运行**，不转换；二元运算遵循类型提升（bf16 + fp32 → fp32） |

对本项目的含义（**这是从规则推出来的假设，必须用 4B.5 的 dtype 实测和 nsys 的 kernel 清单验证**）：
- **残差流大概率是 fp32**：embedding 输出是 fp32，attention 或 FFN 的 bf16 输出加回去时提升为 fp32。
- **每个 Linear 的输入是 fp32，需要先转成 bf16**：每个 Linear 多一个转换 kernel。
- **手写 softmax（max、sub、exp、sum、div）**：`exp` 与 `sum` 会被转成 fp32，其余按输入的 dtype，于是 `attn.softmax` 区间里会多出类型转换 kernel，这一段**不一定变快，甚至可能变慢**。
- **RMSNorm**：若实现内部先转 fp32、算完再转回输入类型，则输出是 fp32，其输入的 dtype 决定了输出。以实测为准。

### 4B.3 测量协议

1. **只改 dtype，其余全部固定**：同一个种子、形状、batch、预热次数、同一个 GPU。
2. 配置开始前显式设置 TF32，并把 `get_float32_matmul_precision()` 的读回值写进结果。
3. 同一个配置依次跑 `float32`、（可选）`tf32`、`bfloat16`。**加一档 tf32 很有用**：fp32 → tf32 的变化是“GEMM 走 Tensor Core”的贡献，tf32 → bf16 的变化是“访存减半加类型转换”的贡献。
4. 每档用阶段计时（4A.3）得到 forward、backward 的均值±std，以及峰值显存。
5. **加速比 = 同一阶段 fp32 时间 ÷ bf16 时间**，在表头注明是哪个阶段（本项目取 forward + backward 之和）。optimizer 不受 autocast 影响（权重和梯度仍是 fp32），所以整步加速比会被 optimizer 稀释。
6. 数值检查：刚初始化时（同种子、同数据）的 loss，fp32 与 bf16 应非常接近（相对差在 1e-2 量级以内）；训练时 loss 必须有限。
7. 峰值显存要说明取的是哪一档（反向结束时，还是完整步）以及单位（GiB 与 GB 差约 7%）。

### 4B.4 为什么加速比达不到理论值（先给假设，再用数据验证）

| 因素 | 方向 | 怎么验证 |
|---|---|---|
| GEMM 从 fp32 的 SIMT 路径换成 bf16 Tensor Core | 加速，通常是最大的一项 | nsys 里 kernel 名是否从 `simt_sgemm` 变成含 `bf16` 或 `tensorop` 的名字（具体名字以实测为准） |
| elementwise 的读写字节减半 | 加速 | ncu 的 DRAM 吞吐不变但字节数减半 |
| 每个 Linear 多一次输入类型转换、每步多一次权重转换 | 减速 | nsys 里 `copy` 类 kernel 的次数与占比上升 |
| softmax 被拆成 bf16 与 fp32 两种精度来回转 | 减速 | `attn.softmax` 区间 kernel 数是否多于 fp32 的 5 个 |
| 小模型、短 ctx 本来就发射受限 | 加速比很小 | GPU 利用率自检：kernel 数多而 kernel 很短，换精度几乎不改变耗时 |
| optimizer 不变 | 稀释整步加速比 | 对比 forward+backward 与整步的加速比 |
显存方面，bf16 不一定省：激活保存的是 bf16 副本（省），但 fp32 权重一直在，autocast 还会缓存一份 bf16 权重（多）。净效果要测。

### 4B.5 dtype 实测（3.5 附加表）

不要凭印象填，用 hook 实测：
```python
seen = {"Linear": [], "RMSNorm": [], "TransformerBlock": []}
for m in model.modules():
    k = type(m).__name__
    if k in seen:
        m.register_forward_hook(lambda m, i, o, k=k: seen[k].append(o.dtype))
with torch.autocast("cuda", dtype=torch.bfloat16):
    logits = model(x)
loss = ce(logits.float().view(-1, logits.shape[-1]), y.view(-1)); loss.backward()
# 参数: p.dtype；梯度: p.grad.dtype；logits、loss: .dtype
```
填表时：

| 对象 | 预期（待实测） |
|---|---|
| 模型参数 | fp32 |
| 第一层 linear 输出 | bf16 |
| LayerNorm / RMSNorm 输出 | 依实现，**以实测为准** |
| logits | bf16 |
| loss | fp32（在 autocast 之外、对 `.float()` 后的 logits 计算） |
| 梯度 | fp32（autograd 把 bf16 副本的梯度转回参数的 dtype） |

注意区分：如果 loss 是在 autocast **之内**直接对 bf16 logits 计算的，`cross_entropy` 在升到 fp32 的名单里，loss 也是 fp32，但路径不同。表里要写明你的计算方式。

---

## 5. 通用模板
```python
import torch, statistics, time
def gpu_time(fn, warmup=5, iters=20, flush_l2=False):
    """每次调用的 GPU 耗时(ms)。微基准用 flush_l2=True。"""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    if flush_l2:
        buf = torch.empty(256 * 2**20, dtype=torch.uint8, device="cuda")  # 须大于 L2
    S = [torch.cuda.Event(enable_timing=True) for _ in range(iters)]
    E = [torch.cuda.Event(enable_timing=True) for _ in range(iters)]
    for i in range(iters):
        if flush_l2:
            buf.zero_()                      # 在 start 之前，不计入
        S[i].record(); fn(); E[i].record()
    torch.cuda.synchronize()                 # 全部记录完再同步，读取才安全
    return [s.elapsed_time(e) for s, e in zip(S, E)]
def wall_ms(fn, warmup=5, iters=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3
def stats(xs):                               # 同时给均值、样本 std、中位数
    med = statistics.median(xs)
    return dict(mean=statistics.fmean(xs), std=statistics.stdev(xs) if len(xs) > 1 else 0.0,
                median=med, min=min(xs), max=max(xs), jitter=(max(xs) - min(xs)) / med)
def phase_step(model, x, y, opt, ce, amp):   # 一步里切出 forward / backward / optimizer
    ev = [torch.cuda.Event(enable_timing=True) for _ in range(4)]
    opt.zero_grad(set_to_none=True)
    ev[0].record()
    with amp():
        logits = model(x)
    loss = ce(logits.float().view(-1, logits.shape[-1]), y.view(-1))
    ev[1].record()
    loss.backward()
    ev[2].record()
    opt.step()
    ev[3].record()
    torch.cuda.synchronize()
    return [ev[i].elapsed_time(ev[i + 1]) for i in range(3)]
```
显存：
```python
fn(); torch.cuda.synchronize()               # 预热之后
torch.cuda.reset_peak_memory_stats()
fn(); torch.cuda.synchronize()
peak_gib = torch.cuda.max_memory_allocated() / 2**30
```

---

## 6. 派生指标

| 指标 | 公式 | 对照 |
|---|---|---|
| 达到算力 | FLOPs / t | 硬件峰值 P（fp32、TF32、bf16 各不同） |
| 达到带宽 | bytes moved / t | 硬件带宽峰值 |
| 吞吐 | B·T / t | tokens/s，用连续墙钟的 t |
| 利用率 | 达到值 ÷ 峰值 | 越接近 1 越没有优化空间 |
| 理想时间 | max(FLOPs/P, bytes/BW) | roofline 下限，**测量值不应低于它** |
| 各阶段 FLOPs 约定 | forward = F；backward ≈ 2F；一步 = 3F（不含 optimizer 和逐元素） | forward 的 TFLOP/s 用 F，整步用 3F 与 step 时间；单独算 backward 的 TFLOP/s 要注明 2F 是近似 |

---

## 7. 控制变量清单
- [ ] 设备独占：`nvidia-smi` 确认空闲，`CUDA_VISIBLE_DEVICES` 隔离
- [ ] 精度一致：TF32 显式设置；bf16 autocast 确认生效
- [ ] 形状固定：`torch.compile` 下形状变化会重编译
- [ ] 输入常驻 GPU：生成与 H2D 在计时外
- [ ] 计时区内无 `.item()`、`.cpu()`、`print`、`.tolist()`（会隐式同步）
- [ ] 模式一致：`train/eval`、`no_grad` 与要比的场景相同
- [ ] 固定种子并保存配置
- [ ] 预热覆盖所有首次开销（含 compile、优化器状态分配）

---

## 8. 统计与抖动
- 报告**中位数**，不是平均值（平均值被偶发慢迭代拉高）。
- **作业要求均值±std 时两个都报**：用样本标准差（n−1）。n 只有 10 左右时，std 本身也不稳，jitter 小（< 5%）时均值与中位数会很接近；若两者差得多，说明有离群值，要先查原因而不是挑一个报。
- 同时报最小值（近似无干扰的最好情况）与最大值（暴露干扰）。
- 抖动 = (max − min) / median，超过约 10% 先查卡是否空闲。
- 次数：单步几十 ms 以上，20 次够；微秒级 kernel 要内层循环摊薄发射开销或用 CUDA graph。
- 同一测量至少跑两遍。

---

## 9. 微基准 vs 宏基准

| | 微基准（单 kernel / 算子） | 宏基准（训练步 / 推理请求） |
|---|---|---|
| 目标 | 一个 kernel 离硬件极限多远 | 端到端真实速度 |
| L2 缓存 | 需要清 | 不用 |
| 发射开销 | 会主导，需摊薄或用 graph | 是真实成本，要保留 |
| 工具 | `do_bench`、`Timer`、ncu | benchmark 脚本、nsys |
| 输入 | 多份轮换，避免缓存命中 | 真实数据 |

---

## 10. 测量本身的正确性检查
先确认测量没问题，再解释结果。

| 检查 | 期望 | 不满足说明 |
|---|---|---|
| 时间不低于理想时间 | 测量值 ≥ roofline 下限 | 漏了同步，或结果没被使用被优化掉 |
| 批大小翻倍（GPU 受限） | 耗时约翻倍 | 几乎不变 → 发射受限或固定开销 |
| 去掉同步再测一遍 | 数字突然极小 | 证明原来的同步有效 |
| iters 20 → 100 | 中位数基本不变 | 未进稳态或有周期性干扰 |
| 两次独立运行 | 中位数差 < 3% | 环境不稳定 |
| 阶段之和 ≈ 整步 | forward + backward + optimizer ≈ step；与连续墙钟（`wall`）相差在几 % 内 | 差得多说明阶段之间有隐式同步或 CPU 空隙 |
| optimizer 只随模型变、不随 ctx 变 | 同一模型不同 ctx 的 optimizer 耗时几乎相等 | 差得多说明测量混入了别的东西 |
| backward ÷ forward 约 2 | 量级在 1.5～2.5 | 远离说明某一阶段的定义不一致（如 forward 含不含 loss） |

---

## 11. 常见坑

| 坑 | 后果 | 处理 |
|---|---|---|
| 没同步就读时间 | 量到发射时间 | event 加 `synchronize` |
| 预热不够 | 前几次拉高统计 | 预热到稳定；compile 后多跑几次 |
| 计时区里 `.item()` | 隐式同步，破坏流水 | 挪到计时区外 |
| 编译、懒加载计入 | 不可复现 | 预热包含它们 |
| 只报平均值 | 被离群值扭曲 | 中位数 + 最小最大（作业要求均值±std 时两个都报） |
| 卡不空闲 | 抖动大 | 先 `nvidia-smi` |
| 优化器状态首次分配 | 第一步显存与耗时偏大 | 预热 ≥ 1 步，预热后再重置峰值 |
| 结果未被使用 | 被消除，时间趋近 0 | 使用或保存输出 |
| fwd 与 train 模式混测 | 不可比 | 统一 `no_grad` 与否 |
| profiler 下的时间当性能 | profiler 有开销 | 时间只信不加 profiler 的 benchmark |
| 训练模式 loss 变 NaN | 数值异常可能改变耗时 | 记录 loss 有限性，降低 lr |
| 对同一个 loss 重复 backward | 报错，或用 `retain_graph` 后显存与行为失真 | 每次迭代重新 forward |
| 用不同运行相减得阶段时间 | backward 被高估、optimizer 误差放大、得不到 std | 同一步里用 event 切段（4A.3） |
| autocast 包住了 backward 或 loss | loss 在 bf16 下计算，或额外转换 | 只包 forward，loss 与 backward 在外面 |
| 把模型参数转成 bf16 再叫“autocast" | 没有 fp32 主权重，训练数值不同，对照组也不对 | 参数保持 fp32；bf16 权重只用于只测推理的大模型，并单独命名 |
| fp32 对照组悄悄走了 TF32 | 加速比被低估 | 每个配置开始前显式设置并记录读回值 |
| 显存基线含 Adam 状态 | 与 `--no-opt` 的 fwd+bwd 峰值差约 8 B/参数 | 表里写明取的是哪一档 |
| OOM 后不清理就跑下一个配置 | 后面的配置也 OOM，或数据失真 | 释放引用、`gc.collect()`、`empty_cache()`，并把 OOM 记进状态列 |
| large 完整训练步放不下 | 以为是脚本 bug | 先按 16 B/参数估算；用 `--no-opt` 补 fwd+bwd |
| `torch.compile` 后形状变化 | 重新编译，计入耗时 | 一个配置一次编译，预热包含编译 |
| GiB 与 GB 混用 | 差约 7% | 表头写单位 |

---

## 12. 记录什么，测完怎么用
**写进 JSON 的内容**：设备名、`torch` 与 CUDA 版本、模型配置、形状、dtype、是否 compile、预热与迭代次数、原始每次耗时、峰值显存、git commit、`get_float32_matmul_precision()` 读回值、每个阶段的 mean/std/median、状态（OK / OOM / 仅 fwd）、是否使用优化器、刚初始化时的 loss、loss 是否有限。
**测完的使用顺序**：
1. 对照预测：比值 0.8 到 1.25 视为预测成立。
2. 超出范围就从粗到细找原因：
   - benchmark → **多快**
   - nsys → **时间花在哪**、GPU 是否在等 CPU
   - ncu → **某个 kernel 为什么慢**（算力还是带宽受限）
3. GPU 利用率高才值得优化 kernel；利用率低先减少 kernel 数或消除发射开销（融合、`torch.compile`、CUDA graph）。

---

## 13. 项目脚本速查（cs336 `benchmark.py`）
**模式**
| 模式 | 内容 |
|---|---|
| `fwd` | `no_grad` 前向（不含 loss），逐次 event |
| `train` | 同一步里切出 forward(+loss) / backward / optimizer 的均值±std；整步；连续墙钟；两档显存探针 |
| `all` | 先 fwd 再 train |

**常用参数**

| 参数 | 作用 |
|---|---|
| `--model a b c`、`--ctx 128 256 ...`、`--dtype float32 bfloat16` | 一次扫多个配置，OOM 记在状态列，不中断 |
| `--no-opt` | 不建优化器，只测 fwd+bwd |
| `--force-train` | xl、2.7b 默认只测 fwd，这个参数让它们也测训练步 |
| `--dtype bf16_weights` | 权重直接转 bf16，用于 2.7B 的 fwd |
| `--check-dtypes` | 打印 autocast 下各类张量的 dtype 表 |
| `--peak-tflops` | 给出则打印相对峰值的利用率 |
| `--warmup 5 --iters 10` | 作业默认值 |

**FLOPs 公式**（forward，不含逐元素）：`L·8·N·d² + L·6·N·d·d_ff + L·4·N·T·d + 2·N·d·V`，其中 `N = B·T`；`train` 的 TFLOP/s 用 3 倍。
small（L=12, d=768, d_ff=3072, V=10000, B=4, T=512）：forward 534.0 GFLOP，一步约 1.60 TFLOP。
**命令**
```bash
# 3.2
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model small medium large --ctx 128 256 512 1024
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model large --ctx 128 256 512 1024 --no-opt
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model xl --ctx 128 --mode fwd
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model 2.7b --ctx 128 --mode fwd --dtype bf16_weights
# 3.5
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model small medium --ctx 512 --dtype float32 tf32 bfloat16
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model large --ctx 256 --dtype float32 bfloat16 --no-opt
CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --check-dtypes --model small --ctx 128 --dtype bfloat16
```
---

## 14. 最小核对清单
- [ ] 被测函数不含数据生成、拷贝、`.item()`、打印
- [ ] 预热足够，编译与分配已完成
- [ ] 用 event 或前后同步计时，没有裸 `time.time()`
- [ ] 报告中位数、最小、最大、抖动（作业另要均值±样本 std）
- [ ] 区分逐次延迟与连续吞吐
- [ ] 每个阶段前重置显存峰值（在预热之后）
- [ ] 通过正确性检查（不低于理想时间、批大小缩放）
- [ ] 配置与环境已保存
- [ ] 与预测对照，差异有归因
- [ ] 训练步各阶段在同一步里用 event 切段，没有用不同运行相减
- [ ] 每次计时迭代都重新 forward（计算图只能用一次）
- [ ] autocast 只包 forward；loss 和 backward 在外面；参数保持 fp32
- [ ] 每个配置开始前显式设置 TF32，并记录读回值
- [ ] OOM 记入状态列，并先按 16 B/参数估算是否应该 OOM
- [ ] dtype 表用 hook 实测，不凭印象
- [ ] 显存表注明取的是哪一档，以及单位
